# End-to-End ML Pipeline Template

**When:** Weeks 13–15 (reuse for capstone)  
**Goal:** One repeatable workflow from raw CSV → evaluated model → saved artifact.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.datasets import load_breast_cancer


## Step 0 — Load data (demo uses sklearn; swap for your CSV)

In [ ]:
# --- Demo data ---
bundle = load_breast_cancer(as_frame=True)
df = bundle.frame.copy()
target_col = "target"

# --- To use a class CSV instead, uncomment and edit: ---
# csv_path = Path("../data_cleaningML/data_cleaning/diabetes.csv")
# df = pd.read_csv(csv_path)
# target_col = "Outcome"  # change to your label column

print(df.shape)
df.head()


## Step 1 — Quick EDA

In [ ]:
print(df[target_col].value_counts(normalize=True).round(3))
print(df.isna().sum().sum(), "missing values total")
df.describe().T.head(10)


## Step 2 — Features / target + preprocessing pipeline

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

print("Numeric:", len(numeric_features), "Categorical:", len(categorical_features))


## Step 3 — Train / compare models

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

candidates = {
    "logreg": LogisticRegression(max_iter=2000),
    "rf": RandomForestClassifier(n_estimators=200, random_state=42),
}

results = {}
for name, model in candidates.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring="f1")
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results[name] = {"cv_f1_mean": cv.mean(), "pipe": pipe, "y_pred": y_pred}
    print(f"\n=== {name} ===")
    print("CV F1:", round(cv.mean(), 3), "+/-", round(cv.std(), 3))
    print(classification_report(y_test, y_pred))


## Step 4 — Select best + plot

In [ ]:
best_name = max(results, key=lambda k: results[k]["cv_f1_mean"])
best_pipe = results[best_name]["pipe"]
print("Best model:", best_name)

ConfusionMatrixDisplay.from_predictions(y_test, results[best_name]["y_pred"])
plt.title(f"Confusion Matrix — {best_name}")
plt.show()


## Step 5 — Save model for Streamlit / demo

In [ ]:
out_dir = Path("../../Deploy/artifacts")
out_dir.mkdir(parents=True, exist_ok=True)
model_path = out_dir / f"best_model_{best_name}.joblib"
joblib.dump(best_pipe, model_path)
print("Saved:", model_path.resolve())


## Capstone checklist
- [ ] Problem statement in plain English
- [ ] Data dictionary / column meanings
- [ ] Cleaning decisions documented
- [ ] At least 2 models compared with the same metric
- [ ] Confusion matrix + short business interpretation
- [ ] Limitations + next steps
- [ ] Saved model artifact
